In [1]:
import xmlrpc.client
import time
from pathlib import Path
from openpyxl import *

NEOS = "https://neos-server.org:3333"   # endpoint NEOS XML-RPC
server = xmlrpc.client.ServerProxy(NEOS)

category = "milp"
solver = "scip"
input_type = "AMPL"

model_text = Path("e-2e-vrp.mod").read_text()
run_text = Path("e-2e-vrp.run").read_text()
set_text = Path("model_set.txt").read_text()

book = "Instances.xlsx"
workbook = load_workbook(book)
sheet = workbook['Hoja1']

In [ ]:
#email = "ca.rodrigueze12@uniandes.edu.co"
#email = "camiloa.rodriguez@javeriana.edu.co"
#email = "camilorodrigueze@outlook.com"
#email = "andreshey25@gmail.com"
email = "andrespinosa2009@hotmail.com"

stop_all = True

for i in range(598):    
    if sheet.cell(2+i, 6).value is None and stop_all:
        instance_name = sheet.cell(2+i, 1).value
        data_text = Path(f"./e-2e-vrp instances/{instance_name}.dat").read_text()
        xml = f"""<document>
        <category>{category}</category>
        <solver>{solver}</solver>
        <inputMethod>{input_type}</inputMethod>
        <email>{email}</email>
        <mod><![CDATA[{model_text}]]></mod>
        <dat><![CDATA[{data_text}]]></dat>
        <com><![CDATA[{run_text}]]></com>
        <par><![CDATA[{set_text}]]></par>
        </document>"""

        try:
            jobNum, password = server.submitJob(xml)

            if jobNum > 0:
                print(f"{instance_name} enviada -> job {jobNum}, pwd {password}")
                sheet.cell(2+i, 6, value=jobNum)
                sheet.cell(2+i, 7, value=password)
                time.sleep(1) 
            else:
                stop_all = False
                workbook.save(book)

        except Exception as e:
            print(f"Error enviando {instance_name}: {e}")
            break

    if sheet.cell(2+i, 8).value == "Error":
        instance_name = sheet.cell(2+i, 1).value
        data_text = Path(f"./e-2e-vrp instances/{instance_name}.dat").read_text()
        xml = f"""<document>
        <category>{category}</category>
        <solver>{solver}</solver>
        <inputMethod>{input_type}</inputMethod>
        <email>{email}</email>
        <mod><![CDATA[{model_text}]]></mod>
        <dat><![CDATA[{data_text}]]></dat>
        <com><![CDATA[{run_text}]]></com>
        <par><![CDATA[{f"{set_text}limits/memory = 3000\nmemory/savefac = 0.7\npresolving/maxrounds = 2\npresolving/abortfac = 0.05"}]]></par>
        </document>"""

        try:
            jobNum, password = server.submitJob(xml)

            if jobNum > 0:
                print(f"{instance_name} enviada -> job {jobNum}, pwd {password}")
                sheet.cell(2+i, 6, value=jobNum)
                sheet.cell(2+i, 7, value=password)
                sheet.cell(2+i, 8).value=None

                time.sleep(1) 
            else:
                stop_all = True
                workbook.save(book)

        except Exception as e:
            print(f"Error enviando {instance_name}: {e}")
            break

workbook.save(book)

breunig-R011-H enviada -> job 18026721, pwd JEXVirju
breunig-R012 enviada -> job 18026722, pwd DEeuVHQG
breunig-R012-H enviada -> job 18026723, pwd YydVoewb
breunig-R013 enviada -> job 18026724, pwd MRraQJGj
breunig-R013-H enviada -> job 18026725, pwd uVCQMsFb
breunig-R038 enviada -> job 18026726, pwd RknejsZq
breunig-R038-H enviada -> job 18026727, pwd uTVvFPLo
breunig-R039 enviada -> job 18026728, pwd PrfFSyOt
breunig-R039-H enviada -> job 18026729, pwd ecnqFldQ
breunig-R040 enviada -> job 18026730, pwd fAiTlcrq
breunig-R040-H enviada -> job 18026731, pwd svLVNljk
breunig-R041 enviada -> job 18026733, pwd biKurLEw
breunig-R041-H enviada -> job 18026734, pwd YsjMaZKe
breunig-R042 enviada -> job 18026735, pwd EyRlzTBd
breunig-R042-H enviada -> job 18026736, pwd oCNJlBdD
breunig-R043 enviada -> job 18026737, pwd kvMPIZrE


In [8]:
import re
from decimal import Decimal

for i in range(598):
    if sheet.cell(2+i, 6).value is not None and sheet.cell(2+i, 8).value is None:
        jobNum = sheet.cell(2+i, 6).value
        password = sheet.cell(2+i, 7).value

        status = server.getJobStatus(jobNum, password)

        if status == "Done":
            try:
                result = server.getFinalResults(jobNum, password).data.decode("utf-8")
                m_time = re.search(r"Solving Time \(sec\)\s*:\s*([0-9.+eE\-]+)", result, re.IGNORECASE)
                m_time = float(m_time.group(1))
                m_primal = re.search(r"Primal Bound\s*:\s*\+?([0-9.+eE\-]+)", result, re.IGNORECASE)
                m_primal = float(Decimal(m_primal.group(1)))
                m_gap = re.search(r"Gap\s*:\s*([0-9.+eE\-]+)\s*%", result, re.IGNORECASE)
                m_gap = float(m_gap.group(1))

                sheet.cell(2+i, 8, value=m_primal)
                sheet.cell(2+i, 9, value=m_time)
                sheet.cell(2+i, 10, value=m_gap)
                
                workbook.save(book)

                print(sheet.cell(2+i, 1).value, f"{m_time} s", m_primal, f"{100*m_gap}%")
            except Exception as e:
                print("Revisar instancia",sheet.cell(2+i, 1).value)
                sheet.cell(2+i, 8, value="Error")
                workbook.save(book)
        else:
            print(sheet.cell(2+i, 1).value, status)

Revisar instancia schneider-R051
Revisar instancia schneider-R051-H
schneider-R001 Running
Revisar instancia schneider-R001-H
Revisar instancia schneider-R004-H
Revisar instancia schneider-R012-H
Revisar instancia schneider-R025-H
schneider-R048 Running
schneider-R048-H Running
Revisar instancia schneider-R083
schneider-R083-H Running
schneider-R019-H Running
Revisar instancia schneider-R034-H
Revisar instancia schneider-R071-H
Revisar instancia schneider-R028
schneider-R028-H Running
Revisar instancia schneider-R045
Revisar instancia schneider-R045-H
Revisar instancia schneider-R002
Revisar instancia schneider-R002-H
Revisar instancia schneider-R006
Revisar instancia schneider-R006-H
Revisar instancia schneider-R023
Revisar instancia schneider-R023-H
Revisar instancia schneider-R035
Revisar instancia schneider-R035-H
Revisar instancia schneider-R078
Revisar instancia schneider-R078-H
Revisar instancia schneider-R085
Revisar instancia schneider-R085-H
Revisar instancia schneider-R056
R